## HuggingFace Transformers: API, Models and Fine-Tuning Techniques
### Day 2: Exploring transformers for inferencing using Task-specific models

---

### Planned agenda for today

#### Module 2: The Transformers API Foundation
- **The Pipeline API: Zero-Shot Inference**
  - Understanding the high-level pipeline abstraction
  - Default models and task-specific pipelines
  - Batch processing with pipelines

- **Tokenizer Fundamentals**
  - Tokenization strategies: BPE, WordPiece, SentencePiece
  - The tokenizer-model relationship
  - Padding, truncation, and attention masks
  - Special tokens and their significance

- **Hands-on Lab**
  - Running inference with pipelines (sentiment analysis, NER, summarization)
  - Tokenization deep dive: encoding/decoding text
  - Comparing tokenizers across model architectures
  - Handling variable-length sequences and batching

#### Module 3: Model Architectures and Task-Specific Models
- **Encoder-Only Models (BERT Family)**
  - Architecture: Bidirectional attention, MLM pre-training
  - Use cases: Classification, NER, Question Answering
  - Understanding hidden states and contextual embeddings
  - Model variants: RoBERTa, DeBERTa, ALBERT, ModernBERT

- **Decoder-Only Models (GPT Family)**
  - Architecture: Causal attention, autoregressive generation
  - Use cases: Text generation, chat, code generation
  - Generation strategies: greedy, beam search, sampling, temperature
  - Model variants: GPT-2, GPT-Neo, Phi, Gemma, SmolLM2

- **Hands-on Lab**
  - Feature extraction with BERT: obtaining embeddings
  - Text generation with GPT models and parameter tuning
  - Comparing generation strategies visually
  - Experimenting with small models suitable for local execution

#### Module 4: Encoder-Decoder and Multimodal Models
- **Encoder-Decoder Models (T5, BART)**
  - Architecture and seq2seq paradigm
  - Use cases: Translation, summarization, text-to-text tasks
  - Task-specific prompting with T5

- **Multimodal Transformers**
  - Vision Transformers (ViT) for image classification
  - CLIP and vision-language models
  - Speech and audio models overview (Whisper)

- **Hands-on Lab**
  - Translation and summarization with T5/BART
  - Image classification with ViT
  - Zero-shot image classification with CLIP
  - Basic speech recognition with Whisper-tiny


---

### Sign in to HuggingFace

In [ ]:
# Load the Hugging Face token from the .env file

from dotenv import load_dotenv
import os
load_dotenv("../../myenv.sh")
HF_TOKEN = os.getenv("HF_TOKEN")

In [ ]:
# Sign in to Hugging Face Hub using the token
from huggingface_hub import login
login(token=HF_TOKEN) 
# NOTE: If the environment variable by name HF_TOKEN is set,
# the login() function will automatically use it, 
# so you can also just call login() without any arguments.

---

### Explore HuggingFace Hub Programmatically


In [ ]:
import huggingface_hub as hf

# List top 10 models sorted by most downloads
for model in hf.list_models(sort="downloads", limit=10):
    print(f"Model: {model.modelId}, Downloads: {model.downloads:,}")

In [ ]:
list_models?

In [ ]:
model

In [ ]:
print(f"{model.id=}")
print(f"{model.modelId=}")
print(f"{model.pipeline_tag=}")
print(f"{model.library_name=}")
print(f"{model.downloads=}")
print(f"{model.likes=}")
print(f"{model.created_at=}")
print(f"{model.tags=}")
print(f"{model.gated=}")
print(f"{model.author=}")
print(f"{model.base_models=}")
print(f"{model.card_data=}")
print(f"{model.config=}")
print(f"{model.children_model_count=}")


In [ ]:
# Find models for a specific task

import huggingface_hub as hf

api = hf.HfApi()

sentiment_models = list(api.list_models(
   filter="text-generation", 
   sort="likes", 
   limit=5
))

for model in sentiment_models:
    print(f"Model: {model.modelId}, Likes: {model.likes:,}, Downloads: {model.downloads:,}")

In [ ]:
for model in hf.list_models(
    filter="llama", 
    sort="downloads", 
    limit=5, 
    gated=False):

    print(f"Model: {model.modelId}, Tags: {model.tags}")

---

### Load and inspect a model


In [ ]:
from transformers import AutoConfig

#model_name = "bert-base-uncased"
model_name = "distilbert-base-uncased-distilled-squad"
config = AutoConfig.from_pretrained(model_name)
print(f"Model type: {config.model_type}")
print(f"Number of hidden layers: {config.num_hidden_layers}")
print(f"Hidden size: {config.hidden_size}")
print(f"Number of attention heads: {config.num_attention_heads}")
print(f"Vocabulary size: {config.vocab_size}")

#### Factory methods in ```transformers```

The ```transformers``` module implements the factory-method design pattern to return a corresponding model, model-config and model-specific tokenizers for most models.

- The ```AutoConfig.from_pretrained(model_name: str)``` returns model config object corresponding to the passed pretrained model name as a string

- The ```AutoModel.from_pretrained(model_name: str)``` returns the corresponding model object with model weights from a pretrained model_name passed as a string

- The ```AutoTokenizer.from_pretrained(model_name: str)``` returns the corresponding tokenizer object suitable to the model passed as model_name (as a string).

NOTE: In most cases, to load model with pretrained model weights, you might want to select a task-specific factory method. For example:
  - ```AutoModelForSequenceClassification``` for classification tasks.
  - ```AutoModelForCausalLM``` for text generation (Generative Autoregressive tasks).
  - ```AutoModelForQuestionAnswering``` for QA related tasks.
  - ```AutoModelForSeq2SeqLM``` for translation, summarization.

The list of task-specific Auto classes are documented at https://huggingface.co/docs/transformers/main/en/model_doc/auto#natural-language-processing

For official tutorial, lookup https://huggingface.co/docs/transformers/models

In [ ]:
config

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"Tokenizer: {tokenizer}")

In [ ]:
from transformers import AutoModel

MODEL_NAME = "ProsusAI/finbert"
model = AutoModel.from_pretrained(MODEL_NAME)
model

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "ProsusAI/finbert"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")
print(f"Model: {model}")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
print(f"Model: {model}")

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, 
                                                           num_labels=2)

model

In [ ]:
# Silence warning about UNEXPECTED keys
import logging

logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
print(f"Tokenizer: {tokenizer}")
print(f"Model: {model}")

In [ ]:

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {total_params:,}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters in the model: {trainable_params:,}")


---

### Tokenizer Deep Dive


In [ ]:
from transformers import AutoTokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer

In [ ]:
#text = "The transformer model is revolutionizing NLP!"

text = "Hi, How are you doing today? I hope you're having a great day!"

# Tokenize -> Encode -> Decode
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")


In [ ]:
text = "Python is an easy language for ML workflow. Python is freely downloadable."
print(tokenizer(text))
print(tokenizer.tokenize(text))


In [ ]:

encoded = tokenizer(text)
print(f"Encoded input IDs: {encoded['input_ids']}")
print(f"Attn Mask: {encoded['attention_mask']}")

decoded = tokenizer.decode(encoded['input_ids'])
print(f"Decoded text: {decoded}")

In [ ]:
# Batch encoding with padding
texts = [
    "Transformers are amazing but complex. Python makes it easy.",
    "Python is cool."
] 

batch_encoded = tokenizer(texts, padding='max_length', max_length=25)
print(batch_encoded)
#print(f"\nBatch shape: {batch_encoded['input_ids'].shape}")
#print(f"Padded IDs:\n{batch_encoded['input_ids']}")

In [ ]:
tokenizer?

---

### Download and Explore a Dataset

In [ ]:
from datasets import load_dataset

# Load IMDB dataset for sentiment analysis
dataset = load_dataset("stanfordnlp/imdb")
dataset

In [ ]:
for d in dataset["train"]:
    if d["label"] == 1:
        print(f"Positive review: {d['text']}")
        break

In [ ]:

print(f"Dataset splits: {dataset.keys()}")
print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

# Explore a sample
sample = dataset['train'][0]
print(f"\nSample text: {sample['text'][:200]}...")
print(f"Label: {sample['label']} ({'Positive' if sample['label'] == 1 else 'Negative'})")

# Check label distribution
from collections import Counter
labels = dataset['train']['label']
print(f"\nLabel distribution: {Counter(labels)}")

# Load a specific subset (faster for experimentation)
small_train = dataset['train'].select(range(1000))
small_test = dataset['test'].select(range(500))
print(f"\nSmall train size: {len(small_train)}")
print(f"Small test size: {len(small_test)}")

---

### Using the ```pipeline()``` from the ```transformers``` library

In [ ]:
from transformers import pipeline

MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline("sentiment-analysis", model=MODEL_NAME)
print(classifier)
print("-" * 40)

result = classifier("I love machine learning!")
print(result)

result = classifier("Transformers are amazing but complex.")
print(result)

result = classifier("I hate bugs in my code.")
print(result)

---

### Pipeline Examples: NLP Tasks


In [ ]:
# 1. Sentiment Analysis
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
results = classifier(["Great product!", "Terrible experience..."])
# [{'label': 'POSITIVE', 'score': 0.99}, {'label': 'NEGATIVE', 'score': 0.98}]
print(results)


In [ ]:
# 2. Named Entity Recognition

# MODEL_NAME = "dbmdz/bert-large-cased-finetuned-conll03-english"

MODEL_NAME = "dslim/bert-base-NER"
ner = pipeline("ner", aggregation_strategy="simple", model=MODEL_NAME)

#text = "Elon Musk founded SpaceX in Hawthorne, California."
text = "Chandrashekar is a corporate trainer and technologist in FOSS who lives in Chennai, India and currently trains for Qualcomm"
entities = ner(text)

print(entities)
for entity in entities:
    print(f"Entity: {entity['word']}, Type: {entity['entity_group']}, Score: {entity['score']:.4f}") 

In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")

In [ ]:
# 3. Question-Answering

# WARNING: "question-answering" pipeline is not implemented as yet
# in HuggingFace transformers 5.x. 
# This code will work on transformers 4.x, but may not work on 5.x.

from transformers import pipeline

MODEL_NAME = "distilbert/distilbert-base-cased-distilled-squad"
qa = pipeline("question-answering", model=MODEL_NAME)

result = qa(
    question="What is the capital of France?",
    context="France is a country in Western Europe. Its capital is Paris."
)
print(result)

print(result["answer"])

In [ ]:
# 4. Summarization

from transformers import pipeline

#MODEL_NAME = "Falconsai/text_summarization"
#MODEL_NAME = "facebook/bart-large-cnn"
#MODEL_NAME = "t5-small"  # Using T5 model for summarization
MODEL_NAME = "sshleifer/distilbart-cnn-12-6"

# device = torch.accelerator.current_accelerator().name
# summarizer = pipeline("summarization", model=MODEL_NAME, device=device)

summarizer = pipeline("summarization", model=MODEL_NAME)
print(summarizer)
text = """The transformer model, introduced in the paper 'Attention Is All You Need',
has revolutionized the field of natural language processing. Unlike previous 
sequence-to-sequence models that relied on recurrent neural networks, the 
transformer uses self-attention mechanisms to process entire sequences 
simultaneously, enabling better parallelization and capturing long-range 
dependencies more effectively."""

summary = summarizer(text, max_length=50, min_length=25)
print(f"Summary: {summary[0]['summary_text']}")

In [ ]:
# 4. Summarization

from transformers import pipeline

#MODEL_NAME = "Falconsai/text_summarization"
#MODEL_NAME = "facebook/bart-large-cnn"
#MODEL_NAME = "t5-small"  # Using T5 model for summarization
MODEL_NAME = "sshleifer/distilbart-cnn-12-6"

# device = torch.accelerator.current_accelerator().name
# summarizer = pipeline("summarization", model=MODEL_NAME, device=device)

summarizer = pipeline("summarization")
print(summarizer)
text = """The transformer model, introduced in the paper 'Attention Is All You Need',
has revolutionized the field of natural language processing. Unlike previous 
sequence-to-sequence models that relied on recurrent neural networks, the 
transformer uses self-attention mechanisms to process entire sequences 
simultaneously, enabling better parallelization and capturing long-range 
dependencies more effectively."""

summary = summarizer(text, max_length=50, min_length=25)
print(f"Summary: {summary[0]['summary_text']}")

---

### Using the ```pipeline()``` from the ```transformers``` library

> NOTE: transformers 5.x has introduced breaking changes in their architecture.

> Thus, many tasks like question-answering / summarization are not available via
> the pipeline() API.  

> Also, many models that are not updated recently would fail to work with transformers 5.x. This includes a lot of ASR, Text-To-Speech and Image generation models.

> The fix is currently underway. Until then, it is better to explore transformers 4.x for our examples where transformers 5.x breaks.

To create a seperate environment for experimenting with transformers 4.x:
```bash
   conda create -n hf4_env python=3.11 jupyter -y
   conda activate hf4_env
   conda install -c conda-forge "transformers>=4.40,<5.0" \
                    datasets evaluate accelerate          \
                    pytorch torchvision torchaudio -y
```

In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")

In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")

> NOTE: If you get SSLVerificationError, try running the following command:
```bash
conda config --set ssl_verify false
```

And then try installing

In [ ]:
import transformers
import torch
import sys
print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)
print("Python version:", sys.version)

print("Accelerator: ", torch.accelerator.current_accelerator())

In [ ]:
from transformers import pipeline

pipeline?

In [ ]:
from transformers import pipeline
classifier = pipeline(task="sentiment-analysis", 
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
result = classifier("I love machine learning!")
print(result)

result = classifier("Transformers are amazing but complex.")
print(result)

result = classifier("I hate bugs in my code.")
print(result)
print(result[0]["label"], result[0]["score"])

---

### Pipeline Examples: NLP Tasks


In [ ]:
# 1. Sentiment Analysis
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
results = classifier(["Great product!", "Terrible experience..."])
# [{'label': 'POSITIVE', 'score': 0.99}, {'label': 'NEGATIVE', 'score': 0.98}]
print(results)


In [ ]:
# Silencing warnings for cleaner output (needed for transformers 5.x)
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

In [ ]:
# 2. Named Entity Recognition (ner)
from transformers import pipeline
ner = pipeline(task="ner", 
               aggregation_strategy="simple", 
               model="dbmdz/bert-large-cased-finetuned-conll03-english")

#text = "Elon Musk founded SpaceX in Hawthorne, California."
text = "Chandrashekar is a managing director of Slashprog Technologies, lives in Chennai, India and trains for Qualcomm."
entities = ner(text)
print("---- Named Entities ---")
print(entities)


In [ ]:
for entity in entities:
    print(f"Entity: {entity['word']}, Type: {entity['entity_group']}, Score: {entity['score']:.4f}") 

In [ ]:
p = pipeline("question-answering", model="distilbert/distilbert-base-cased-distilled-squad")
p

In [ ]:
pipeline?

In [ ]:
from transformers import pipeline

#MODEL_NAME = "gpt2"
#MODEL_NAME = "Qwen/Qwen3-0.6B"
MODEL_NAME = "google/gemma-3-1b-it"
text_gen = pipeline("text-generation", model=MODEL_NAME)
result = text_gen("The distance between Earth and Moon is", max_new_tokens=50)
print(result[0]["generated_text"])

In [ ]:
from huggingface_hub import list_models

qa_models = list_models(filter="question-answering", sort="downloads", limit=5)
for model in qa_models:
    print(f"Model: {model.modelId}, Downloads: {model.downloads:,}")

In [ ]:
# 3. Text-Generation

text_gen = pipeline("text-generation", model='openai-community/gpt2')

result = text_gen(
    "What is the distance between the Earth and the Moon in kilometers?"
)

#print(result)
print(result[0]['generated_text'])


---

### Using the low-level API for more tasks

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# pipe = pipeline("text-generation", model=model_name)

model_inputs = tokenizer("This distance between the Earth and the Moon in kilometers is", return_tensors="pt")
generated_ids = model.generate(**model_inputs, max_new_tokens=20)
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(generated_text)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openai-community/gpt2"
#model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer


In [ ]:

model = AutoModelForCausalLM.from_pretrained(model_name)
model

In [ ]:

text = "This distance between the Earth and the Moon in kilometers is"
model_inputs = tokenizer(text, return_tensors="pt")

model_inputs


In [ ]:

generated_ids = model.generate(**model_inputs, max_new_tokens=50)
generated_ids.squeeze()


In [ ]:
generated_text = tokenizer.decode(generated_ids.squeeze())
print(generated_text)


---

### Question-Answering without ```pipeline()``` API

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

MODEL_NAME = "distilbert-base-uncased-distilled-squad"
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

question = "How many parameters does BLOOM support?"
context = "BLOOM has 176 billion parameters and can generate text in 46 languages natural languages and 13 programming languages."

inputs = tokenizer(question, context, return_tensors="pt")
inputs


In [ ]:

outputs = model(**inputs)
outputs


In [ ]:
s = outputs.start_logits.argsort(descending=True).squeeze()[1]
e = outputs.end_logits.argsort(descending=True).squeeze()[1]
print(s, e)
inputs["input_ids"].squeeze()[s:e + 1]
t = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze()[s:e + 1])
t

In [ ]:
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

In [ ]:
i = inputs["input_ids"].squeeze()[answer_start_index:answer_end_index + 1]
i

In [ ]:
t = tokenizer.convert_ids_to_tokens(i)
t

In [ ]:
tokenizer.convert_tokens_to_string(t)

In [ ]:

answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()
answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start_index:answer_end_index+1]))
print(answer)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
tokenizer?

---

### Question-Answering using the ```pipeline()``` API

In [ ]:
from transformers import pipeline

#model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")
#tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")


context = """The Linux kernel project is a free and open-source, monolithic, Unix-like operating system kernel. 
The Linux kernel was conceived and created in 1991 by Linus Torvalds for his personal computer, and it has since 
grown to support a wide variety of computer architectures, including the x86, ARM, and RISC-V instruction sets. 
The Linux kernel is released under the GNU General Public License version 2 (GPLv2), which allows anyone to view, 
modify, and distribute the source code. The kernel is developed by a large community of developers from around 
the world, with contributions from individuals, companies, and organizations. It is used as the foundation for 
many operating systems, including popular distributions such as Ubuntu, Fedora, and Debian.
"""

#question = "Who created the Linux kernel?"
#question = "When was the Linux kernel created?"
#question = "What is the license of the Linux kernel?"
#question = "What is the Linux kernel used for?"
#question = "What is the Linux kernel?"
question = "Which are popular distributions that use the Linux kernel?"

qa_pipeline = pipeline(task="question-answering", 
                       model="distilbert-base-cased-distilled-squad", 
                       tokenizer="distilbert-base-cased-distilled-squad")
answer = qa_pipeline(question=question, context=context)
print(answer["answer"])
print(answer)


In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline

#model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")
#tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")

qa_pipeline = pipeline("question-answering", 
                       model="distilbert-base-uncased-distilled-squad", 
                       tokenizer="distilbert-base-uncased-distilled-squad")

context = """The Linux kernel project is a free and open-source, monolithic, Unix-like operating system kernel. 
The Linux kernel was conceived and created in 1991 by Linus Torvalds for his personal computer, and it has since 
grown to support a wide variety of computer architectures, including the x86, ARM, and RISC-V instruction sets. 
The Linux kernel is released under the GNU General Public License version 2 (GPLv2), which allows anyone to view, 
modify, and distribute the source code. The kernel is developed by a large community of developers from around 
the world, with contributions from individuals, companies, and organizations. It is used as the foundation for 
many operating systems, including popular distributions such as Ubuntu, Fedora, and Debian.
"""

questions = [
    "Who created the Linux kernel?",
    "When was the Linux kernel created?",
    "What is the license of the Linux kernel?",
    "What is the Linux kernel used for?",
    "What is the Linux kernel?",
    "Which are popular distributions that use the Linux kernel?"
]

for question in questions:
    answer = qa_pipeline(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {answer['answer']}\n")
    
    #inputs = tokenizer(question, context, return_tensors="pt")
    #outputs = model(**inputs)
    #answer_start_index = outputs.start_logits.argmax()
    #answer_end_index = outputs.end_logits.argmax()
    #answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start_index:answer_end_index+1]))
    #print(f"Q: {question}")
    #print(f"A: {answer}\n")


---

### Text Generation Example

In [ ]:
# Text Generation example using DeepSeek-R1-Distill-Qwen-1.5B model
from transformers import pipeline
#from accelerate import Accelerator
#device = Accelerator().device

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
text_gen = pipeline("text-generation", model=model_name)

result = text_gen("The distance between the Earth and the Moon is approximately", 
                   temperature=1.2)

print(result[0]['generated_text'])

In [ ]:
# Text Generation example without using pipeline()
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

model = AutoModelForCausalLM.from_pretrained(model_name")

tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "The distance between the earth and the moon in kilometers is approximately"
model_inputs = tokenizer(text, return_tensors="pt").to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens=500, temperature=0.2, do_sample=True)
result = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(result)

In [ ]:
# Text Generation example using Gemma-3-1B-IT model
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator

device = Accelerator().device

model_name = "google/gemma-3-1b-it"

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "The distance between the earth and the moon in kilometers is approximately"
model_inputs = tokenizer(text, return_tensors="pt").to(device)

generated_ids = model.generate(**model_inputs, temperature=0.6, do_sample=True)
result = tokenizer.decode(generated_ids[0])
print(result)

In [ ]:
# Basic Prompt Engineering Example

from transformers import pipeline

text_gen = pipeline("text-generation", model="google/gemma-3-1b-it")

prompt = """Classify each of the following texts into one of the following categories: Science, Philosophy, History, Geography.

Text: The distance between the Earth and the Moon is approximately 384,400 kilometers.
Text: The Zen principles emphasize meditation and intuition rather than ritual worship or study of scriptures.
Text: The Great Wall of China is a series of fortifications built along the northern borders of China.
Text: Delhi is the capital of India and is known for its rich history and cultural heritage.

"""

outputs = text_gen(prompt, temperature=0.7, max_new_tokens=200, do_sample=True, num_return_sequences=1)
for output in outputs:
    print(output['generated_text'])
    print("\n---\n")

In [ ]:
# Basic Prompt Engineering Example

from transformers import pipeline

text_gen = pipeline("text-generation", model="google/gemma-3-1b-it")

prompt = """Classify each of the following texts into one of the following categories: Science, Philosophy, History, Geography.

Text: The distance between the Earth and the Moon is approximately 384,400 kilometers.
Text: The Zen principles emphasize meditation and intuition rather than ritual worship or study of scriptures.
Text: The Great Wall of China is a series of fortifications built along the northern borders of China.
Text: Delhi is the capital of India and is known for its rich history and cultural heritage.

"""

outputs = text_gen(prompt, temperature=0.9, max_new_tokens=200, do_sample=True, num_return_sequences=1)
for output in outputs:
    print(output['generated_text'])
    print("\n---\n")

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", 
                model="google/gemma-3-1b-it", 
                dtype="auto")

messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are Ed Sheeran"},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on AI using Python Language"},]
        },
    ],
]

output = pipe(messages, max_new_tokens=500)
print(output)
print("-" * 30)
print(output[0][0]['generated_text'][-1]["content"])

---
### Sentiment Analysis

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

classifier(["Python + ML skills = Super-human capabilities",
           "World Models are the future of AI",
           "Building neural networks using Assembly language can be painful"])

---

### Text Summarization example

In [ ]:
# Text Summarization Example
# pipeline() does not work with "summarization" task on transformers 5.x.
# So, we need to initialize the model and tokenizer separately.

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

from accelerate import Accelerator
device = Accelerator().device

# import torch
# device = torch.accelerator.current_accelerator().name


model_name = "facebook/bart-large-cnn"

text = """
America has changed dramatically during recent years. Not only has the number of
graduates in traditional engineering disciplines such as mechanical, civil,
electrical, chemical, and aeronautical engineering declined, but in most of
the premier American universities engineering curricula now concentrate on
and encourage largely the study of engineering science. As a result, there
are declining offerings in engineering subjects dealing with infrastructure,
the environment, and related issues, and greater concentration on high
technology subjects, largely supporting increasingly complex scientific
developments. While the latter is important, it should not be at the expense
of more traditional engineering.

Rapidly developing economies such as China and India, as well as other
industrial countries in Europe and Asia, continue to encourage and advance
the teaching of engineering. Both China and India, respectively, graduate
six and eight times as many traditional engineers as does the United States.
Other industrial countries at minimum maintain their output, while America
suffers an increasingly serious decline in the number of engineering graduates
and a lack of well-educated engineers.
"""

summarizer = pipeline(task="summarization", 
                      model=model_name, 
                      tokenizer=model_name)

print("Summary:")
result = summarizer(text, max_length=150, min_length=40, do_sample=False)
print(result)
print("-" * 30)

summary = result[0]['summary_text']

print(summary.replace(". ", ".\n"))


In [ ]:
# Text Summarization Example
# pipeline() does not work with "summarization" task on transformers 5.x.
# This code should work with transformers 5.x.
# So, we need to initialize the model and tokenizer separately.

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from accelerate import Accelerator
device = Accelerator().device

model_name = "facebook/bart-large-cnn"

text = """
America has changed dramatically during recent years. Not only has the number of
graduates in traditional engineering disciplines such as mechanical, civil,
electrical, chemical, and aeronautical engineering declined, but in most of
the premier American universities engineering curricula now concentrate on
and encourage largely the study of engineering science. As a result, there
are declining offerings in engineering subjects dealing with infrastructure,
the environment, and related issues, and greater concentration on high
technology subjects, largely supporting increasingly complex scientific
developments. While the latter is important, it should not be at the expense
of more traditional engineering.

Rapidly developing economies such as China and India, as well as other
industrial countries in Europe and Asia, continue to encourage and advance
the teaching of engineering. Both China and India, respectively, graduate
six and eight times as many traditional engineers as does the United States.
Other industrial countries at minimum maintain their output, while America
suffers an increasingly serious decline in the number of engineering graduates
and a lack of well-educated engineers.
"""

model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(device)

generated_ids = model.generate(**inputs,  
                               num_beams=4, 
                               length_penalty=0.9, 
                               no_repeat_ngram_size=3, 
                               early_stopping=True)
summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("Summary:")
print(summary.replace(". ", ".\n"))


---

### Text to Speech synthesis example

In [ ]:
# Text to Audio synthesis Example

from transformers import pipeline
from accelerate import Accelerator
device = Accelerator().device

#model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice" # Does not work with pipeline() on transformers 5.x
model_name = "suno/bark-small"
tts = pipeline("text-to-speech", model=model_name, device=device)

print("Generating audio from text...")

text = "This is a sample text to demonstrate the text-to-speech synthesis capabilities of the model."

audio = tts(text)
print(audio)

# The Audio can be played directly in a Jupyter Notebook as below:
from IPython.display import Audio
Audio(audio["audio"], rate=audio["sampling_rate"])

In [ ]:
from IPython.display import Audio
Audio(audio["audio"], rate=audio["sampling_rate"])

In [ ]:
# Suitable for transformers 5.x, using AutoProcessor and BarkModel directly
import torch
from transformers import AutoProcessor, BarkModel
from accelerate import Accelerator

device = Accelerator().device
model_name = "suno/bark-small"
# Load the processor and model explicitly
processor = AutoProcessor.from_pretrained(model_name)
model = BarkModel.from_pretrained(model_name).to(device)

text = "This is a sample text to demonstrate the text-to-speech synthesis capabilities of the model."

# 1. Manually prepare inputs with attention_mask
inputs = processor(text, return_tensors="pt").to(device)

print("Generating audio tokens...")

# 2. Feed attention_mask and define pad_token_id to eliminate the warning
audio_tokens = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=processor.tokenizer.eos_token_id  # Force padding to map to EOS
)
# Convert generated audio tokens back to a numpy array for playback/saving
audio_array = audio_tokens.cpu().numpy().squeeze()

print("Audio generation complete.")

from IPython.display import Audio
Audio(audio_array, rate=16000)  # Assuming a sample rate of 16k

In [ ]:
from transformers import pipeline

tts_pipeline = pipeline("text-to-speech", model="facebook/mms-tts-eng")

text_prompt = "The Hugging Face pipeline makes audio generation incredibly easy."
output = tts_pipeline(text_prompt)
Audio(output["audio"], rate=output["sampling_rate"])

---

### Automatic Speech Recognition (ASR) Example

In [ ]:
pwd

In [ ]:
# Automatic Speech Recognition (ASR) Example

from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="distil-whisper/distil-small.en")
asr_result = asr("../../harvard.wav")
asr_result["text"]

In [ ]:
# Automatic Speech Recognition (ASR) Example

from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="distil-whisper/distil-small.en")
asr_result = asr("../../my_voice.wav", return_timestamps=True)
print(asr_result["text"])



In [ ]:
asr_result

---

### Text to Image Generation Example

In [ ]:
!conda install diffusers -c conda-forge -y

In [ ]:
# Text to Image Generation Example
import torch
from diffusers import StableDiffusionPipeline
from accelerate import Accelerator
device = Accelerator().device

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16
)
pipe = pipe.to(device)

prompt = "a photorealistic image of an astronaut riding a horse in a futuristic city, digital art"
result = pipe(prompt)
print(result)
image = result.images[0]
image.save("../astronaut_riding_horse3.png")
image.show()

In [ ]:
from diffusers import DiffusionPipeline
import torch

from accelerate import Accelerator
device = Accelerator().device

model_name = "Qwen/Qwen-Image"

pipe = DiffusionPipeline.from_pretrained(model_name, torch_dtype=torch.float16)
pipe = pipe.to(device)

positive_magic = {
    "en": ", Ultra HD, 4K, cinematic composition.", # for english prompt
    "zh": ", 超清，4K，电影级构图." # for chinese prompt
}

# Generate image
prompt = '''
A coffee shop entrance features a chalkboard sign reading 
"Qwen Coffee 😊 $2 per cup," with a neon light beside it 
displaying "通义千问". Next to it hangs a poster showing a beautiful 
Chinese woman, and beneath the poster is 
written "π≈3.1415926-53589793-23846264-33832795-02384197". 
Ultra HD, 4K, cinematic composition'''

negative_prompt = " " # using an empty string if you do not have specific concept to remove


# Generate with different aspect ratios
aspect_ratios = {
    "1:1": (1328, 1328),
    "16:9": (1664, 928),
    "9:16": (928, 1664),
    "4:3": (1472, 1140),
    "3:4": (1140, 1472),
    "3:2": (1584, 1056),
    "2:3": (1056, 1584),
}

width, height = aspect_ratios["16:9"]

image = pipe(
    prompt=prompt + positive_magic["en"],
    negative_prompt=negative_prompt,
    width=width,
    height=height,
    num_inference_steps=50,
    true_cfg_scale=4.0,
    generator=torch.Generator(device=device).manual_seed(42)
).images[0]

image.save("../qwen_image_example.png")


---

### Image Segmentation using ViT

In [ ]:
from transformers import pipeline
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/segmentation_input.jpg"
image = Image.open(requests.get(url, stream=True).raw)
image

In [ ]:
from transformers import pipeline
semantic_segmentation = pipeline("image-segmentation", 
                                 "nvidia/segformer-b1-finetuned-cityscapes-1024-1024")

results = semantic_segmentation(image)
results

In [ ]:
print(results[9]["label"], results[9]["score"], results[9]["mask"])

---

## Encoder-Decoder use-cases

### Text Translation

In [ ]:
!conda install -c conda-forge sentencepiece -y

In [ ]:
from transformers import pipeline

# Helsinki-NLP models for 1000+ language pairs
translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-fr"
)
result = translator("Hello, how are you?")
# [{'translation_text': 'Bonjour, comment allez-vous?'}]

print(result[0]['translation_text'])


##### Language Code Formats:
- Helsinki-NLP: ISO 639-1 (`en`, `fr`, `de`, `zh`)
- NLLB: FLORES-200 codes (`eng_Latn`, `fra_Latn`, `zho_Hans`)
- M2M-100: Similar to FLORES codes


In [ ]:

# Multilingual model (NLLB - No Language Left Behind)
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",  # Source language code
    tgt_lang="hin_Deva"   # Target language code
)
result = translator("Hello, how are you?")
print(result[0]['translation_text'])

#### Summarization Example

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

article = """The transformer is a deep learning architecture introduced 
in the 2017 paper 'Attention Is All You Need' by Google researchers. 
Unlike previous models that processed sequences sequentially, the 
transformer uses self-attention mechanisms to process all tokens 
simultaneously, enabling much better parallelization. This breakthrough 
has led to significant advances in natural language processing, 
including models like BERT, GPT, and T5. The architecture has since 
been adapted for computer vision, speech recognition, and multimodal 
applications."""

summary = summarizer(
    article,
    max_length=50,
    min_length=20,
    do_sample=False
)
print(summary[0]['summary_text'])
# "The transformer architecture uses self-attention to process 
#  sequences in parallel. This has led to advances in NLP models 
#  like BERT, GPT, and T5."

---

### Text-to-Text Generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")

# Grammar correction
input_text = "grammar: He have been working here since five years."
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(**inputs, max_length=128)
corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(corrected)
# "He has been working here for five years."

# Question generation from context
context = """The Eiffel Tower is a wrought-iron lattice tower on the 
Champ de Mars in Paris, France. It is named after the engineer 
Gustave Eiffel, whose company designed and built the tower."""
input_text = f"generate question: {context}"
# Output: "Who designed the Eiffel Tower?"
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(**inputs, max_length=128)
question = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(question)


---

## Hands-on Lab: Feature Extraction with BERT (15 mins)



---
### Exercise 1: Getting Contextual Embeddings


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Sample sentences
sentences = [
    "I love programming in Python.",
    "The snake python is a non-venomous constrictor.",
    "Python is both a programming language and a snake species."
]

# Tokenize
inputs = tokenizer(sentences, padding=True, truncation=True, 
                   return_tensors="pt")

# Get embeddings
with torch.no_grad():
    outputs = model(**inputs)

# Extract embeddings
last_hidden_states = outputs.last_hidden_state  # [3, seq_len, 768]
cls_embeddings = last_hidden_states[:, 0, :]     # [3, 768]

# Show "Python" token embedding differs based on context!
for i, sent in enumerate(sentences):
    tokens = tokenizer.tokenize(sent)
    if 'python' in tokens:
        idx = tokens.index('python') + 1  # +1 for [CLS]
        print(f"Sentence: {sent}")
        print(f"  python embedding shape: {last_hidden_states[i, idx].shape}")
        print(f"  First 5 values: {last_hidden_states[i, idx, :5].tolist()}\n")


---

### Lab Exercise 2: Cosine Similarity of Embeddings


In [ ]:
import torch.nn.functional as F

# Get word embeddings for "python" in different contexts
def get_word_embedding(model, tokenizer, sentence, word):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    
    tokens = tokenizer.tokenize(sentence)
    word_tokens = tokenizer.tokenize(word)
    
    # Find position (handle subword tokens)
    for i in range(len(tokens) - len(word_tokens) + 1):
        if tokens[i:i+len(word_tokens)] == word_tokens:
            # Average embeddings if word is split
            return outputs.last_hidden_state[0, i+1:i+1+len(word_tokens)].mean(0)
    return None

# Compare "python" across sentences
emb1 = get_word_embedding(model, tokenizer, 
                          "I love programming in Python.", "python")
emb2 = get_word_embedding(model, tokenizer, 
                          "The python snake is dangerous.", "python")
emb3 = get_word_embedding(model, tokenizer, 
                          "I code in Python.", "python")

# Compute similarities
sim_12 = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0))
sim_13 = F.cosine_similarity(emb1.unsqueeze(0), emb3.unsqueeze(0))
sim_23 = F.cosine_similarity(emb2.unsqueeze(0), emb3.unsqueeze(0))

print(f"Similarity: Programming Python vs Snake Python: {sim_12.item():.3f}")
print(f"Similarity: Programming Python vs Code Python: {sim_13.item():.3f}")
print(f"Similarity: Snake Python vs Code Python: {sim_23.item():.3f}")
print("\nNote: Context matters! Same word, different embeddings.")



---

### Lab Exercise 3: Text Generation with GPT-2


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load GPT-2
model_name = "gpt2"  # 124M parameters - runs on CPU!
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Add padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token

# Test different generation strategies
prompt = "In a distant future, artificial intelligence"

strategies = {
    "Greedy": {
        "do_sample": False, "max_length": 60, "num_beams": 1
    },
    "Beam Search (k=5)": {
        "do_sample": False, "max_length": 60, "num_beams": 5,
        "early_stopping": True
    },
    "Sampling (temp=0.7)": {
        "do_sample": True, "max_length": 60, "temperature": 0.7,
        "top_k": 0
    },
    "Sampling (temp=1.0, top_p=0.9)": {
        "do_sample": True, "max_length": 60, "temperature": 1.0,
        "top_p": 0.9, "top_k": 50
    }
}

for strategy, params in strategies.items():
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, **params)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*50}")
    print(f"{strategy}:")
    print(f"{'='*50}")
    print(generated)
    print()



---

### Lab Exercise 4: Comparing Generation Parameters


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

prompt = "Once upon a time"

# Generate with different temperatures
temperatures = [0.1, 0.3, 0.5, 0.7, 0.9, 1.2, 1.5]

print("Effect of Temperature on Generation:\n")
for temp in temperatures:
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_length=50,
        do_sample=True,
        temperature=temp,
        top_k=50,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Temperature {temp}: {generated}\n")

# Observe:
# - Low temp: Repetitive, deterministic
# - Medium temp: Coherent, natural
# - High temp: Creative but potentially incoherent



---

### Lab Exercise 5: Comparing Model Sizes


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# Compare different model sizes (if available memory)
models_to_test = [
    ("gpt2", "GPT-2 (124M)"),
    ("distilgpt2", "DistilGPT-2 (82M)"),
]

for model_name, display_name in models_to_test:
    try:
        print(f"\nLoading {display_name}...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)
        
        prompt = "The future of AI is"
        inputs = tokenizer(prompt, return_tensors="pt")
        
        # Time the generation
        start = time.time()
        outputs = model.generate(
            **inputs,
            max_length=50,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
        elapsed = time.time() - start
        
        params = sum(p.numel() for p in model.parameters())
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        print(f"Parameters: {params:,}")
        print(f"Generation time: {elapsed:.2f}s")
        print(f"Generated: {generated[:100]}...")
        
        # Clean up memory
        del model, tokenizer
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"  Failed: {e}")



---

# Hands-on Lab: Translation and Summarization (15 mins)

---

## Exercise 1: Multi-Language Translation


In [ ]:
from transformers import pipeline
import torch

# Load translation pipeline
translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-fr",
    device=0 if torch.cuda.is_available() else -1
)

# Translate sentences
english_texts = [
    "The weather is beautiful today.",
    "Machine learning is transforming industries.",
    "I would like a cup of coffee, please."
]

for text in english_texts:
    result = translator(text)
    print(f"EN: {text}")
    print(f"FR: {result[0]['translation_text']}\n")

# Try other language pairs
translators = {
    "en→fr": "Helsinki-NLP/opus-mt-en-fr",
    "en→de": "Helsinki-NLP/opus-mt-en-de",
    "en→es": "Helsinki-NLP/opus-mt-en-es",
    "en→it": "Helsinki-NLP/opus-mt-en-it",
    "en→nl": "Helsinki-NLP/opus-mt-en-nl",
}

text = "Hello, how are you?"
for lang_pair, model_name in translators.items():
    t = pipeline("translation", model=model_name)
    result = t(text)
    print(f"{lang_pair}: {result[0]['translation_text']}")



---

### Lab Exercise 2: Summarization with BART


In [ ]:
from transformers import pipeline

# Load summarization pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1  # Use CPU if GPU memory limited
)

# Sample long text
long_text = """
Artificial intelligence has made remarkable progress in recent years, 
particularly in the field of natural language processing. Large language 
models like GPT-3, BERT, and their successors have demonstrated impressive 
capabilities in understanding and generating human-like text. These models 
are trained on massive datasets containing billions of words from the 
internet, books, and other sources. Through a process called pre-training, 
they learn patterns in language, grammar, and even some reasoning abilities. 

However, these models also have significant limitations. They can generate 
plausible-sounding but factually incorrect information, exhibit biases 
present in their training data, and require enormous computational resources 
to train and deploy. Researchers are actively working on making these models 
more efficient, accurate, and aligned with human values. The field continues 
to evolve rapidly, with new architectures and training methods being 
developed regularly.
"""

# Generate summaries with different lengths
for length_ratio in [0.3, 0.5, 0.7]:
    # Estimate max_length based on input
    input_length = len(long_text.split())
    max_len = int(input_length * length_ratio)
    min_len = max_len // 2
    
    summary = summarizer(
        long_text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )
    print(f"\nSummary ({length_ratio:.0%} length, ~{max_len} words):")
    print(summary[0]['summary_text'])


---

### Lab Exercise 3: Image Classification with ViT


In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

# Load image classification pipeline
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

# Load a sample image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))

# Display image
plt.imshow(image)
plt.axis('off')
plt.show()

# Classify
results = classifier(image)
for result in results[:5]:
    print(f"{result['label']}: {result['score']:.3f}")

# Try with your own images
# image = Image.open("path/to/your/image.jpg")
# results = classifier(image)



---

### Lab Exercise 4: Zero-Shot with CLIP


In [ ]:
from transformers import pipeline

# Load zero-shot image classification pipeline
classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32"
)

# Load image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
import requests
from PIL import Image
from io import BytesIO

response = requests.get(url)
image = Image.open(BytesIO(response.content))

# Define candidate labels (not limited to training labels!)
candidate_labels = [
    "a cat",
    "a dog",
    "a tiger",
    "a lion",
    "a rabbit",
    "a car",
    "a building",
    "food"
]

# Classify
results = classifier(image, candidate_labels=candidate_labels)
for result in results:
    print(f"{result['label']}: {result['score']:.3f}")

# Try with creative/abstract labels
creative_labels = [
    "a majestic feline",
    "a internet meme star",
    "a sleepy animal",
    "a predator",
    "someone's pet"
]

results = classifier(image, candidate_labels=creative_labels)
print("\nCreative labeling:")
for result in results:
    print(f"{result['label']}: {result['score']:.3f}")


---

### Lab Exercise 5: Speech Recognition with Whisper


In [ ]:
from transformers import pipeline
import torch

# Load ASR pipeline (use tiny model for quick testing)
transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=0 if torch.cuda.is_available() else -1
)

# If you have an audio file:
# audio_file = "path/to/audio.mp3"
# result = transcriber(audio_file)
# print(f"Transcription: {result['text']}")

# For testing without audio file - create a note about setup
print("To test speech recognition:")
print("1. Install required packages:")
print("   pip install librosa soundfile")
print()
print("2. Use with audio file:")
print('   result = transcriber("audio.mp3")')
print('   print(result["text"])')
print()
print("Whisper models available:")
print("  whisper-tiny:   39M parameters (fastest)")
print("  whisper-base:   74M parameters")
print("  whisper-small:  244M parameters")
print("  whisper-medium: 769M parameters")
print("  whisper-large:  1.5B parameters (best quality)")

# Demo with a sample from HuggingFace dataset
from datasets import load_dataset

print("\nLoading sample from LibriSpeech...")
try:
    # Load a sample
    dataset = load_dataset("librispeech_asr", "clean", split="test", streaming=True)
    sample = next(iter(dataset))
    
    # Transcribe
    result = transcriber(sample["audio"]["array"])
    print(f"Transcription: {result['text']}")
    print(f"Reference: {sample['text']}")
except Exception as e:
    print(f"Couldn't load sample: {e}")
    print("This is expected if you don't have audio dependencies installed.")


---

### Lab Exercise 6: Multi-Model Pipeline Chain


In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# Create a chain: Image → Caption → Translation
print("Creating multi-model pipeline chain...")

# 1. Image captioning
captioner = pipeline(
    "image-to-text",
    model="nlpconnect/vit-gpt2-image-captioning"
)

# 2. Translation (English → French)
translator = pipeline(
    "translation_en_to_fr",
    model="Helsinki-NLP/opus-mt-en-fr"
)

# 3. Sentiment analysis on caption
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Process an image through the chain
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))

print("\n1. Generating caption...")
caption = captioner(image)[0]['generated_text']
print(f"   Caption: {caption}")

print("\n2. Translating to French...")
translated = translator(caption)[0]['translation_text']
print(f"   French: {translated}")

print("\n3. Analyzing sentiment of caption...")
sentiment = sentiment_analyzer(caption)[0]
print(f"   Sentiment: {sentiment['label']} (confidence: {sentiment['score']:.3f})")

print("\n" + "="*50)
print("Pipeline Chain Complete!")
print(f"  Input: Image of a cat")
print(f"  → Caption: {caption}")
print(f"  → Translation: {translated}")
print(f"  → Sentiment: {sentiment['label']}")

